In [35]:
!pip install -q -U mlflow boto3 awscli dagshub

In [36]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

try:
    
    access_key = user_secrets.get_secret("AWS_ACCESS_KEY")
    secret_key = user_secrets.get_secret("AWS_SECRET_KEY")
    
    # Set environment variables
    os.environ['AWS_ACCESS_KEY_ID'] = access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_key
    os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
    
    print("✅ AWS Credentials successfully loaded!")
    print(f"Access Key ID: {access_key[:4]}****{access_key[-4:]}")
    print(f"Region: {os.environ['AWS_DEFAULT_REGION']}")

except Exception as e:
    print("❌ Failed to find secrets. Make sure you added them in Add-ons -> Secrets.") 

✅ AWS Credentials successfully loaded!
Access Key ID: AKIA****LQZB
Region: us-east-1


In [37]:
## Setup Mlflow Server

mlflow.set_tracking_uri("http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/")
# Set an experiment
mlflow.set_experiment("Exp 3 - TfIdf Trigram max_features")

2026/05/13 16:07:46 INFO mlflow.tracking.fluent: Experiment with name 'Exp 3 - TfIdf Trigram max_features' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://amzn-mlflow-bucket-26/3', creation_time=1778688466599, experiment_id='3', last_update_time=1778688466599, lifecycle_stage='active', name='Exp 3 - TfIdf Trigram max_features', tags={}, trace_location=None, workspace='default'>

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [39]:
df = pd.read_csv('/kaggle/working/reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [40]:
# Step 1: Function to run the experiment
def run_experiment_tfidf_max_features(max_features):
    ngram_range = (1, 3)  # Trigram setting

    # Step 2: Vectorization using TF-IDF with varying max_features
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"TFIDF_Trigrams_max_features_{max_features}")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, max_features={max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, max_features={max_features}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(
            sk_model=model, 
            artifact_path=f"model_tfidf_trigrams_{max_features}"
        )

# Step 6: Test various max_features values
max_features_values = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

for max_features in max_features_values:
    run_experiment_tfidf_max_features(max_features)

2026/05/13 16:08:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 16:08:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TFIDF_Trigrams_max_features_1000 at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3/runs/15e4697ede274c4fa302dddabdcbb1f9
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3


2026/05/13 16:08:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 16:08:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TFIDF_Trigrams_max_features_2000 at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3/runs/228be777a502454d8526dd59b6d46199
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3


2026/05/13 16:09:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 16:09:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run TFIDF_Trigrams_max_features_8000 at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3/runs/4af4efe74291470f99d93f7f4e827eec
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3


2026/05/13 16:12:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 16:12:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TFIDF_Trigrams_max_features_9000 at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3/runs/670b3963a123479da92c0cd379e8cc3a
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3


2026/05/13 16:12:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 16:13:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run TFIDF_Trigrams_max_features_10000 at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3/runs/1c54f406dc0e4eb99f7343449c741591
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/3
